# Title Optimization Analysis

This notebook analyzes the results of the iterative title generation process. It evaluates the effectiveness of the LLM-driven optimization by testing several hypotheses related to performance gains, semantic exploration, and optimization trajectories.

In [ ]:
# Setup and Dependencies
!pip install -q sentence-transformers

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import drive

drive.mount('/content/drive')

RESULTS_PATH = '/content/drive/MyDrive/numeric_inference_outputs/title_optimization_results.json'
EMBEDDING_MODEL_NAME = 'all-MiniLM-L6-v2'

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

with open(RESULTS_PATH, 'r') as f:
    results = json.load(f)

df = pd.DataFrame(results)
print(f"Loaded {len(df)} optimization results.")

## Hypothesis 1: Lower starting performance leads to greater potential for improvement

### Methodology
We compare the predicted log-views of the original title (baseline) against the absolute improvement achieved after 5 iterations. We use Pearson correlation to determine if there is a significant negative relationship between the starting point and the gain.

### Hypotheses
- **Null Hypothesis (H0)**: There is no correlation between the original title's predicted performance and the improvement gained through optimization.
- **Alternative Hypothesis (H1)**: There is a negative correlation; titles with lower initial scores show larger improvements.

### Results

In [ ]:
plt.figure(figsize=(10, 6))
sns.regplot(data=df, x='original_score', y='improvement')

r, p = stats.pearsonr(df['original_score'], df['improvement'])
plt.title(f'Original Performance vs. Improvement\nPearson r: {r:.3f}, p-value: {p:.4e}')
plt.xlabel('Original Predicted Log-Views')
plt.ylabel('Improvement (Log-Views)')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Mean Improvement: {df['improvement'].mean():.4f}")
print(f"Max Improvement: {df['improvement'].max():.4f}")
print(f"Min Improvement: {df['improvement'].min():.4f}")

### Interpretation
If the correlation is significantly negative (p < 0.05), it suggests a 'ceiling effect' or 'regression to the mean' where titles that are already performing well have less room for optimization compared to those that start with poor semantic alignment with channel success drivers.

## Hypothesis 2: Greater semantic exploration leads to higher relative improvement

### Methodology
We calculate the semantic distance (1 - Cosine Similarity) between the original title and the best optimized title. We then correlate this distance with the improvement achieved. This tests whether 'bolder' changes in the title's meaning lead to better outcomes.

### Hypotheses
- **Null Hypothesis (H0)**: Semantic distance from the original title does not correlate with performance improvement.
- **Alternative Hypothesis (H1)**: Greater semantic distance from the original title correlates positively with performance improvement.

### Results

In [ ]:
# Calculate embeddings for distance analysis
orig_titles = df['original_title'].tolist()
best_titles = df['best_optimized_title'].tolist()

orig_embs = embedding_model.encode(orig_titles)
best_embs = embedding_model.encode(best_titles)

distances = []
for i in range(len(df)):
    sim = cosine_similarity(orig_embs[i].reshape(1, -1), best_embs[i].reshape(1, -1))[0][0]
    distances.append(1 - sim)

df['semantic_distance'] = distances

plt.figure(figsize=(10, 6))
sns.regplot(data=df, x='semantic_distance', y='improvement')

r, p = stats.pearsonr(df['semantic_distance'], df['improvement'])
plt.title(f'Semantic Distance vs. Improvement\nPearson r: {r:.3f}, p-value: {p:.4e}')
plt.xlabel('Semantic Distance (1 - Cosine Similarity)')
plt.ylabel('Improvement (Log-Views)')
plt.grid(True, alpha=0.3)
plt.show()

### Interpretation
A positive correlation indicates that the optimization process benefits from exploring regions of the semantic space farther from the original title. This would suggest that the LLM is effectively 'pivoting' the content to better match high-performance patterns.

## Hypothesis 3: Semantic optimization translates to performance via targeted hill-climbing

### Methodology
To test if the model is effectively 'learning' from feedback, we analyze the semantic trajectory across iterations. For each iteration $i$, we measure the average distance of its suggestions to the **best** performing suggestion of iteration $i-1$ vs. the **worst** performing suggestion of iteration $i-1$. 

We expect the suggestions in iteration $i$ to gravitate towards previous 'successes' and move away from previous 'failures'.

### Hypotheses
- **Null Hypothesis (H0)**: Subsequent suggestions do not show a semantic bias toward previous high-performing titles.
- **Alternative Hypothesis (H1)**: Suggestions in later iterations are semantically closer to previous best-performers than to previous worst-performers.

### Results

In [ ]:
trajectory_data = []

for idx, row in df.iterrows():
    history = row['history']
    for i in range(1, len(history)): # Start from iteration 2 (index 1)
        prev_titles = history[i-1]['titles']
        prev_titles_sorted = sorted(prev_titles, key=lambda x: x['score'], reverse=True)
        
        best_prev_text = prev_titles_sorted[0]['text']
        worst_prev_text = prev_titles_sorted[-1]['text']
        
        curr_titles = [t['text'] for t in history[i]['titles']]
        
        # Embeddings
        best_prev_emb = embedding_model.encode([best_prev_text])
        worst_prev_emb = embedding_model.encode([worst_prev_text])
        curr_embs = embedding_model.encode(curr_titles)
        
        # Distances
        dist_to_best = 1 - cosine_similarity(curr_embs, best_prev_emb).flatten()
        dist_to_worst = 1 - cosine_similarity(curr_embs, worst_prev_emb).flatten()
        
        for d_b, d_w in zip(dist_to_best, dist_to_worst):
            trajectory_data.append({
                'video_id': row['video_id'],
                'iteration': i + 1,
                'dist_to_best_prev': d_b,
                'dist_to_worst_prev': d_w,
                'relative_proximity': d_w - d_b # Positive means closer to best than to worst
            })

traj_df = pd.DataFrame(trajectory_data)

plt.figure(figsize=(12, 6))
sns.boxplot(data=traj_df, x='iteration', y='relative_proximity')
plt.axhline(0, color='red', linestyle='--')
plt.title('Semantic Bias: Proximity to Best vs. Worst Previous Suggestions')
plt.ylabel('Distance Difference (Worst - Best)')
plt.xlabel('Iteration')
plt.grid(True, alpha=0.3)
plt.show()

print("Mean Relative Proximity by Iteration (Positive = Closer to Best):")
print(traj_df.groupby('iteration')['relative_proximity'].mean())

In [ ]:
plt.figure(figsize=(10, 6))
sns.kdeplot(data=traj_df, x='dist_to_best_prev', label='Distance to Best Prev', fill=True)
sns.kdeplot(data=traj_df, x='dist_to_worst_prev', label='Distance to Worst Prev', fill=True)
plt.title('Distribution of Semantic Distances to Previous Best/Worst')
plt.xlabel('Distance (1 - Cosine Similarity)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

t_stat, p_val = stats.ttest_rel(traj_df['dist_to_best_prev'], traj_df['dist_to_worst_prev'])
print(f"Paired t-test between dist_to_best and dist_to_worst:")
print(f"t-statistic: {t_stat:.4f}, p-value: {p_val:.4e}")

### Interpretation
The boxplot shows the distribution of `(Distance to Worst) - (Distance to Best)`. Values above 0 indicate the suggestion is semantically closer to the previous iteration's best performer. 

If the distribution is significantly shifted above zero (confirmed by the t-test), it proves that the LLM is successfully hill-climbing in the semantic space, using the feedback to converge on high-performance clusters.